# Phase 3: Feature Engineering

## Advanced Feature Creation for Predictive Analytics

This phase transforms raw data into sophisticated business metrics that power the predictive models used in Phase 5.

### Features to Create
1. **Behavioral Features** - Customer engagement patterns
2. **Financial Features** - Wealth and balance patterns  
3. **Tenure Features** - Loyalty indicators
4. **Composite Scores** - Multi-factor risk indicators
5. **Categorical Encodings** - Segment assignments

---

## Objective
Create features that capture hidden patterns and make the data ready for machine learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/cleaned_data_with_features.csv')
print(f"Data loaded: {df.shape}")

## 1. Behavioral Features

In [ ]:
# Behavioral Features
print("=" * 80)
print("CREATING BEHAVIORAL FEATURES")
print("=" * 80)

# Activity Score (normalized 0-100)
df['Activity_Score'] = df['IsActiveMember'] * 100

# Multi-Product Engagement (binary indicator)
df['Multi_Product_Customer'] = (df['NumOfProducts'] >= 2).astype(int)

# Credit Card Adoption
df['Has_Credit_Card'] = df['HasCrCard']

# Activity x Products (interaction)
df['Activity_Product_Interaction'] = df['IsActiveMember'] * df['NumOfProducts']

# Engagement Tier (categorical to numeric)
engagement_tier_map = {
    'Active Engaged': 3,
    'Active Single Product': 2,
    'Inactive Multi-Product': 1,
    'Inactive Disengaged': 0
}
df['Engagement_Tier'] = df['Engagement_Category'].map(engagement_tier_map)

print("\nBehavioral Features Created:")
print(f"  ✓ Activity_Score")
print(f"  ✓ Multi_Product_Customer")
print(f"  ✓ Has_Credit_Card")
print(f"  ✓ Activity_Product_Interaction")
print(f"  ✓ Engagement_Tier")

## 2. Financial Features

In [ ]:
# Financial Features
print("\n" + "=" * 80)
print("CREATING FINANCIAL FEATURES")
print("=" * 80)

# Balance categories
df['Balance_Category'] = pd.cut(df['Balance'], 
                               bins=[0, 50000, 100000, 200000, np.inf],
                               labels=['Low', 'Medium', 'High', 'Very_High'])
df['Balance_Category_Numeric'] = pd.factorize(df['Balance_Category'])[0]

# Salary categories
df['Salary_Category'] = pd.cut(df['EstimatedSalary'],
                              bins=[0, 50000, 100000, 150000, np.inf],
                              labels=['Low', 'Medium', 'High', 'Very_High'])
df['Salary_Category_Numeric'] = pd.factorize(df['Salary_Category'])[0]

# Balance-to-Salary Ratio (wealth indicator)
df['Balance_To_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)

# High Balance Flag
df['High_Balance'] = (df['Balance'] > 100000).astype(int)

# High Salary Flag
df['High_Salary'] = (df['EstimatedSalary'] > df['EstimatedSalary'].quantile(0.75)).astype(int)

# Premium Customer (high balance AND high salary)
df['Premium_Customer'] = ((df['Balance'] > 100000) & 
                          (df['EstimatedSalary'] > df['EstimatedSalary'].quantile(0.75))).astype(int)

# Wealth Risk Score (high balance + inactive)
df['Wealth_Risk_Score'] = df['High_Balance'] * (1 - df['IsActiveMember'])

print("\nFinancial Features Created:")
print(f"  ✓ Balance_Category")
print(f"  ✓ Salary_Category")
print(f"  ✓ Balance_To_Salary_Ratio")
print(f"  ✓ High_Balance")
print(f"  ✓ High_Salary")
print(f"  ✓ Premium_Customer")
print(f"  ✓ Wealth_Risk_Score")

## 3. Tenure and Loyalty Features

In [ ]:
# Tenure Features
print("\n" + "=" * 80)
print("CREATING TENURE & LOYALTY FEATURES")
print("=" * 80)

# Tenure categories
df['Tenure_Category'] = pd.cut(df['Tenure'],
                              bins=[0, 1, 3, 6, 10],
                              labels=['New', 'Growing', 'Established', 'Veteran'])
df['Tenure_Category_Numeric'] = pd.factorize(df['Tenure_Category'])[0]

# Long tenure flag
df['Long_Tenure'] = (df['Tenure'] >= 5).astype(int)

# Very new customer
df['Very_New'] = (df['Tenure'] <= 1).astype(int)

# Tenure-Activity Interaction (loyalty indicator)
df['Tenure_Activity_Score'] = (df['Tenure'] / 10) * df['IsActiveMember'] * 100

# Customer Lifespan Potential Score
df['Lifespan_Potential'] = (df['Tenure'] * df['IsActiveMember'] * 
                            (1 + df['NumOfProducts']/4) * 100).astype(float)

print("\nTenure & Loyalty Features Created:")
print(f"  ✓ Tenure_Category")
print(f"  ✓ Long_Tenure")
print(f"  ✓ Very_New")
print(f"  ✓ Tenure_Activity_Score")
print(f"  ✓ Lifespan_Potential")

## 4. Composite Risk & Value Scores

In [ ]:
# Composite Scores
print("\n" + "=" * 80)
print("CREATING COMPOSITE RISK & VALUE SCORES")
print("=" * 80)

# Churn Risk Score (0-100, higher = more at risk)
df['Churn_Risk_Score'] = (
    (1 - df['IsActiveMember']) * 40 +  # Inactivity = 40%
    (1 - df['Multi_Product_Customer']) * 30 +  # Single product = 30%
    (1 - df['Has_Credit_Card']) * 15 +  # No card = 15%
    (df['Very_New']) * 15  # Very new = 15%
)

# Customer Value Score (0-100, higher = more valuable)
df['Customer_Value_Score'] = (
    df['Tenure_Category_Numeric'] * 20 +  # Tenure weight
    df['High_Balance'] * 30 +  # Balance weight
    df['Premium_Customer'] * 20 +  # Premium weight
    (1 - df['Very_New']) * 30  # Not new = valuable
)

# Retention Propensity Score (0-100, higher = more likely to stay)
df['Retention_Propensity'] = (
    df['IsActiveMember'] * 40 +
    df['Multi_Product_Customer'] * 25 +
    df['Has_Credit_Card'] * 20 +
    (df['Tenure']/10 * 15)  # Tenure normalized to 0-15
)

# Customer Segmentation based on Value and Risk
def segment_customer(row):
    if row['Customer_Value_Score'] >= 60 and row['Churn_Risk_Score'] <= 30:
        return 'VIP_Loyal'
    elif row['Customer_Value_Score'] >= 60 and row['Churn_Risk_Score'] > 30:
        return 'VIP_At_Risk'
    elif row['Customer_Value_Score'] < 60 and row['Churn_Risk_Score'] <= 30:
        return 'Standard_Loyal'
    else:
        return 'Standard_At_Risk'

df['Customer_Segment'] = df.apply(segment_customer, axis=1)

print("\nComposite Scores Created:")
print(f"  ✓ Churn_Risk_Score (0-100)")
print(f"  ✓ Customer_Value_Score (0-100)")
print(f"  ✓ Retention_Propensity (0-100)")
print(f"  ✓ Customer_Segment (4 categories)")

print("\n\nScore Distributions:")
print(f"\nChurn Risk Score:")
print(df['Churn_Risk_Score'].describe())
print(f"\nCustomer Value Score:")
print(df['Customer_Value_Score'].describe())
print(f"\nRetention Propensity:")
print(df['Retention_Propensity'].describe())

In [ ]:
# Feature Summary
print("\n" + "=" * 80)
print("FINAL FEATURE SUMMARY")
print("=" * 80)

# Count new features
original_cols = 14  # From original dataset
new_cols = len(df.columns) - original_cols

print(f"\nOriginal columns: {original_cols}")
print(f"New features created: {new_cols}")
print(f"Total columns: {len(df.columns)}")

print(f"\n\nAll Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# Save enhanced data
output_path = '../data/data_with_engineered_features.csv'
df.to_csv(output_path, index=False)
print(f"\n\n✓ Enhanced data saved to: {output_path}")

# Segment distribution
print("\n\nCustomer Segment Distribution:")
print(df['Customer_Segment'].value_counts())
print("\n\nChurn Rate by Segment:")
print(df.groupby('Customer_Segment')['Exited'].agg(['count', lambda x: f'{x.mean():.2%}']))
